<a href="https://colab.research.google.com/github/ingridmidory/Escuela-Nacional-Preparatoria-No.4-Vidal-Casta-eda-y-N-jera-/blob/main/Horarios_Prepa_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import zipfile
import glob
import os

In [ ]:
df = pd.read_excel("/content/Horarios 25-26.xlsx")

In [ ]:
# ==============================
# CONFIGURACIÓN
# ==============================

# Mapear días y horas
dias_map = {"LU":"Lunes","MA":"Martes","MI":"Miércoles","JU":"Jueves","VI":"Viernes"}
horas_map = {
    1:"07:00–07:50", 2:"07:50–08:40", 3:"08:40–09:30", 4:"09:30–10:20",
    5:"10:20–11:10", 6:"11:10–12:00", 7:"12:00–12:50", 8:"12:50–13:40",
    9:"13:40–14:30", 10:"14:30–15:20", 11:"15:20–16:10", 12:"16:10–17:00",
    13:"17:00–17:50", 14:"17:50–18:40", 15:"18:40–19:30", 16:"19:30–20:20",
    17:"20:20–21:10", 18:"21:10-22:00"
}
dias = list(dias_map.keys())
modulos = list(range(1,19))

dias = list(dias_map.keys())
modulos = list(range(1,19))

In [ ]:
# ==============================
# NORMALIZAR GRUPOS
# ==============================
def normalizar_grupo(grupo):
    return ''.join(filter(str.isdigit, str(grupo)))  # 401A -> 401

df['GRUPO_NORMAL'] = df['GRUPO'].apply(normalizar_grupo)

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 655.9 kB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def generar_pdf_horarios_horizontal(tabla, nombre_archivo, titulo):
    # Crear PDF tamaño Carta horizontal
    pdf = SimpleDocTemplate(nombre_archivo, pagesize=landscape(letter),
                            topMargin=40, bottomMargin=40,
                            leftMargin=30, rightMargin=30)

    styles = getSampleStyleSheet()
    style_title = styles["Title"]
    style_title.fontSize = 14
    style_title.alignment = 1  # centrado

    elementos = []
    elementos.append(Paragraph(titulo, style_title))
    elementos.append(Spacer(1, 20))

    # Convertir cada celda en Paragraph para wrap automático
    data = [["Hora"] + [Paragraph(str(dias_map[d]), styles["Normal"]) for d in tabla.columns]]
    for i, row in enumerate(tabla.values):
        fila = [Paragraph(str(horas_map[tabla.index[i]]), styles["Normal"])]
        fila += [Paragraph(str(val), styles["Normal"]) for val in row]
        data.append(fila)

    t = Table(data, repeatRows=1)

    n_rows = len(data)
    n_cols = len(data[0])

    # Calcular ancho y alto de celdas según espacio disponible en landscape
    page_width, page_height = landscape(letter)
    usable_width = page_width - 60  # márgenes left+right
    usable_height = page_height - 80  # márgenes top+bottom y espacio título

    ancho_col = usable_width / n_cols
    altura_fila = usable_height / n_rows

    # Ajustar tamaño de fuente dinámicamente si hay muchas filas
    fontsize = 10
    if n_rows > 20:
        fontsize = max(6, 10 - (n_rows - 20)//5)

    t._argW = [ancho_col] * n_cols
    t._argH = [altura_fila] * n_rows

    # Estilo de tabla
    style = TableStyle([
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('FONTSIZE', (0,0), (-1,-1), fontsize),
        ('BACKGROUND', (0,0), (-1,0), colors.lightblue),  # fila encabezado superior
        ('BACKGROUND', (0,0), (0,-1), colors.lightblue),  # columna de horas
    ])
    t.setStyle(style)

    elementos.append(t)
    pdf.build(elementos)


In [ ]:
# ==============================
# GENERAR HORARIOS POR GRUPO
# ==============================
os.makedirs("horarios_grupos", exist_ok=True)

for grupo, datos in df.groupby("GRUPO_NORMAL"):
    tabla = pd.DataFrame("", index=modulos, columns=dias)
    for _, row in datos.iterrows():
        dia, hora = row["HORA"][:2], int(row["HORA"][2:])
        texto = f"{row['MATERIA']}\n{row['SALON']}"
        if tabla.loc[hora, dia] != "":
            tabla.loc[hora, dia] += "\n---\n" + texto  # combinar optativas
        else:
            tabla.loc[hora, dia] = texto
    generar_pdf_horarios(tabla, f"horarios_grupos/horario_grupo_{grupo}.pdf", f"Horario Grupo {grupo}")

In [ ]:
# ==============================
# GENERAR HORARIOS POR SALÓN
# ==============================
#os.makedirs("horarios_salones", exist_ok=True)

#for salon, datos in df.groupby("SALON"):
#    tabla = pd.DataFrame("", index=modulos, columns=dias)
#    for _, row in datos.iterrows():
#        dia, hora = row["HORA"][:2], int(row["HORA"][2:])
#        texto = f"{row['GRUPO']}\n{row['MATERIA']}"
#        if tabla.loc[hora, dia] != "":
#            tabla.loc[hora, dia] += "\n---\n" + texto
#        else:
#            tabla.loc[hora, dia] = texto
#    generar_pdf_horarios(tabla, f"horarios_salones/horario_salon_{salon}.pdf", f"Horario Salón {salon}", fontsize=10)

In [ ]:
# ==============================
# CREAR ARCHIVOS ZIP
# ==============================
with zipfile.ZipFile("horarios_grupos.zip", 'w') as zipf:
    for pdf in glob.glob("horarios_grupos/*.pdf"):
        zipf.write(pdf)

with zipfile.ZipFile("horarios_salones.zip", 'w') as zipf:
    for pdf in glob.glob("horarios_salones/*.pdf"):
        zipf.write(pdf)

print("✅ Horarios generados y comprimidos en horarios_grupos.zip y horarios_salones.zip")

✅ Horarios generados y comprimidos en horarios_grupos.zip y horarios_salones.zip
